In [ ]:
import os
import json
import random
from pathlib import Path
import numpy as np
import torch
from datasets import load_dataset
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments, DataCollatorWithPadding
import matplotlib.pyplot as plt

# 1. Project Setup
PROJECT_ROOT = Path.cwd()
MODEL_DIR = PROJECT_ROOT / 'saved_model'
MODEL_DIR.mkdir(exist_ok=True)

LABELS = ['World', 'Sports', 'Business', 'Sci/Tech']

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)

# 2. Load Dataset (PATCHED: Using the namespaced repository)
dataset = load_dataset('fancyzhx/ag_news')
print('Train samples:', len(dataset['train']))
print('Test samples:', len(dataset['test']))
print('Classes:', {i: label for i, label in enumerate(LABELS)})
print('Example record:', dataset['train'][0])

# 3. Tokenization
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

def tokenize_batch(batch):
    return tokenizer(
        batch['text'],
        padding='max_length',
        truncation=True,
        max_length=128,
        return_tensors='pt',
    )

# Selecting subsets for quick training/testing
train_dataset = dataset['train'].select(range(2000))
test_dataset = dataset['test'].select(range(500))

train_tokenized = train_dataset.map(tokenize_batch, batched=True, batch_size=1000)
test_tokenized = test_dataset.map(tokenize_batch, batched=True, batch_size=1000)

train_tokenized.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])
test_tokenized.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])

# 4. Model Setup & Training Arguments
model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=len(LABELS))
model.config.id2label = {i: label for i, label in enumerate(LABELS)}
model.config.label2id = {label: i for i, label in enumerate(LABELS)}

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    preds = np.argmax(predictions, axis=1)
    return {
        'accuracy': accuracy_score(labels, preds),
        'precision': precision_score(labels, preds, average='weighted', zero_division=0),
        'recall': recall_score(labels, preds, average='weighted', zero_division=0),
        'f1': f1_score(labels, preds, average='weighted', zero_division=0),
    }

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

training_args = TrainingArguments(
    output_dir=str(PROJECT_ROOT / 'models'),
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=1,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    logging_steps=20,
    report_to='none',
)

# Crucial missing step: Instantiating the Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=test_tokenized,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

# Start training
trainer.train()

# 5. Evaluation & Saving
eval_results = trainer.evaluate()
print("Evaluation Results:", eval_results)

model.save_pretrained(MODEL_DIR)
tokenizer.save_pretrained(MODEL_DIR)

with open(MODEL_DIR / 'label_mapping.json', 'w', encoding='utf-8') as handle:
    json.dump({i: label for i, label in enumerate(LABELS)}, handle, indent=2)

# 6. Prediction Loop & Metrics Visualization
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

texts = test_dataset['text']
labels = test_dataset['label']
preds = []

with torch.no_grad():
    for start in range(0, len(texts), 16):
        batch_texts = texts[start:start+16]
        batch_inputs = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=128,
            return_tensors='pt',
        ).to(device)
        outputs = model(**batch_inputs)
        probs = torch.softmax(outputs.logits, dim=1)
        preds.extend(torch.argmax(probs, dim=1).cpu().tolist())

preds = np.array(preds)
labels = np.array(labels)

print('Accuracy:', accuracy_score(labels, preds))
print('Precision:', precision_score(labels, preds, average='weighted', zero_division=0))
print('Recall:', recall_score(labels, preds, average='weighted', zero_division=0))
print('F1:', f1_score(labels, preds, average='weighted', zero_division=0))
print(classification_report(labels, preds, target_names=LABELS))

# Plotting Confusion Matrix
cm = confusion_matrix(labels, preds)
plt.figure(figsize=(6, 5))
plt.imshow(cm, interpolation='nearest', cmap='Blues')
plt.title('Confusion Matrix')
plt.colorbar()
plt.xticks(np.arange(len(LABELS)), LABELS, rotation=45)
plt.yticks(np.arange(len(LABELS)), LABELS)
plt.ylabel('True label')
plt.xlabel('Predicted label')

for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(j, i, cm[i, j], ha='center', va='center', color='black')

plt.tight_layout()
plt.show()

# 7. Singe Prediction Function
def predict_headline(headline):
    encoded = tokenizer(
        headline,
        padding=True,
        truncation=True,
        max_length=128,
        return_tensors='pt',
    ).to(device)
    with torch.no_grad():
        outputs = model(**encoded)
        probs = torch.softmax(outputs.logits, dim=1)[0]
    confidence = probs.max().item()
    predicted_index = torch.argmax(probs).item()
    predicted_label = LABELS[predicted_index]
    return predicted_label, confidence

headline = 'Apple announces new AI features for iPhone'
print(predict_headline(headline))

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 18.6MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

In [5]:
!pip install --upgrade huggingface_hub transformers datasets

  Using cached transformers-5.13.1-py3-none-any.whl.metadata (32 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 770.3/770.3 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 52.8 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface-hub 0.24.7
    Uninstalling huggingface-hub-0.24.7:
      Successfully uninstalled huggingface-hub-0.24.7
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.21.4
    Uninstalling tokenizers-0.21.4:
      Successfully uninstalled tokenizers-0.21.4
  Attempting un